# Extract Sample .bin Files

Extract example GENEActiv `.bin` files from the **GENEAread** R package and save them to `data/raw/bin/` for pipeline testing. Verify readability with both R (GENEAread) and Python (actipy).

In [4]:
# --- Config ---
import subprocess
from pathlib import Path

R_PATH = r"C:\Program Files\R\R-4.5.3\bin\x64\Rscript.exe"
PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
BIN_OUTPUT_DIR = PROJECT_ROOT / "data" / "raw" / "bin"

# Ensure output dir exists
BIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"R path: {R_PATH}")
print(f"Output dir: {BIN_OUTPUT_DIR}")

R path: C:\Program Files\R\R-4.5.3\bin\x64\Rscript.exe
Output dir: c:\Users\astri\Desktop\Data_Scientist\Projects\veerleproject\data\raw\bin


In [5]:
# --- Extract sample .bin file from GENEAread R package ---
r_script = f"""
bin_dir <- system.file("binfile", package = "GENEAread")
files <- list.files(bin_dir, pattern = "\\\\.bin$", full.names = TRUE)
if (length(files) == 0) {{
  cat("ERROR: No .bin files found in GENEAread package\\n")
  quit(status = 1)
}}
out_dir <- normalizePath("{BIN_OUTPUT_DIR.as_posix()}", mustWork = FALSE)
for (f in files) {{
  dest <- file.path(out_dir, basename(f))
  ok <- file.copy(f, dest, overwrite = TRUE)
  cat(if (ok) "Copied:" else "FAILED:", basename(f), "->", dest, "\\n")
}}
cat("\\nDone. Total files copied:", sum(file.exists(file.path(out_dir, basename(files)))), "\\n")
"""

result = subprocess.run(
    [R_PATH, "--vanilla", "-e", r_script],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

Copied: TESTfile.bin -> C:\Users\astri\Desktop\Data_Scientist\Projects\veerleproject\data\raw\bin/TESTfile.bin 

Done. Total files copied: 1 



In [6]:
# --- Verify the .bin file is readable ---
# Note: actipy requires Java (not installed). Using GENEAread (R) for verification instead.

bin_files = list(BIN_OUTPUT_DIR.glob("*.bin"))
print(f"Found {len(bin_files)} .bin file(s) in {BIN_OUTPUT_DIR}:\n")

for bf in bin_files:
    print(f"  {bf.name}  ({bf.stat().st_size:,} bytes)")

# Verify with GENEAread
if bin_files:
    verify_script = f"""
    library(GENEAread)
    f <- "{bin_files[0].as_posix()}"
    cat("Reading:", f, "\\n")
    dat <- read.bin(f, calibrate = TRUE)
    cat("Class:", class(dat), "\\n")
    cat("Duration:", dat$page.timestamps[length(dat$page.timestamps)] - dat$page.timestamps[1], "seconds\\n")
    cat("Sampling frequency:", dat$freq, "Hz\\n")
    cat("Data dimensions:", nrow(dat$data.out), "rows x", ncol(dat$data.out), "cols\\n")
    cat("Columns:", paste(names(dat$data.out), collapse = ", "), "\\n")
    cat("\\nFirst few rows:\\n")
    print(head(dat$data.out))
    """
    result = subprocess.run([R_PATH, "--vanilla", "-e", verify_script], capture_output=True, text=True)
    print("\n--- GENEAread verification ---")
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)

Found 1 .bin file(s) in c:\Users\astri\Desktop\Data_Scientist\Projects\veerleproject\data\raw\bin:

  TESTfile.bin  (398,302 bytes)

--- GENEAread verification ---
Reading: c:/Users/astri/Desktop/Data_Scientist/Projects/veerleproject/data/raw/bin/TESTfile.bin 
Number of pages in binary file: 104 
Calculated page references... 
Processing...
Processing took: 0.027 secs .
Loaded 31200 records (Approx  2 MB of RAM)
12-05-23 16:47:50.000 (Wed)  to  12-05-23 16:53:01.990 (Wed) 
Class: AccData 
Duration: 5.15 seconds
Sampling frequency: 100 Hz
Data dimensions: 31200 rows x 7 cols
Columns:  

First few rows:
      timestamp             x          y           z light button temperature
[1,] 1337791670  0.0235164141 -0.8872826 -0.10078524     0      0        25.8
[2,] 1337791670 -0.0001578283 -1.0882876 -0.09293286     0      0        25.8
[3,] 1337791670  0.0235164141 -1.0419018 -0.07330192     0      0        25.8
[4,] 1337791670  0.0116792929 -1.0650947 -0.06544955     0      0        25.8
[